# Employee Engagement Analysis Using Machine Learning

**Case Study No. 90**

This notebook runs **Exploratory Data Analysis (EDA)**, builds a
**student-defined engagement target**, then trains and compares classifiers
on the IBM HR table stored in `data/`.

The IBM HR Analytics dataset does **not** contain a column named Employee Engagement.
The classification goal of this case study still needs a Low / Medium / High label,
so later cells construct that label from five existing 1–4 survey items.

## What this notebook does

- Load a **copy** of the CSV (the original file is never overwritten)
- Report dimensions, missing values, and duplicates
- Plot numerical, categorical, and engagement-related distributions
- Screen outliers with the IQR rule
- Show correlations and group comparisons
- Write an **EDA Findings** section from the loaded table only
- Verify the five survey columns exist
- Add `EngagementScore` and `EngagementLevel` **without dropping** the original items
- Show the target distribution, percentages, and class imbalance

## What this notebook does later

After EDA and target construction, later cells train four classifiers on a
stratified split, compare **macro F1** (not accuracy alone), save Joblib
pipelines, and compute Decision Tree Gini importance.

## What this notebook does not do

- Does not invent rows if the CSV is missing
- Does not overwrite the original file in `data/`
- Does not treat accuracy as the only ranking metric
- No causal claims from correlations or feature importance
- Does **not** treat the new score as an official HR or clinical engagement scale

## Dataset file

Expected file: `data/IBM_HR_Attrition_Enhanced.csv`

## 1. Import libraries and project helpers

The notebook lives in `notebooks/`, so the project root is added to `sys.path`.
Plot style and EDA functions come from `src/eda.py` so every chart is reproducible.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Find the project root whether the kernel cwd is notebooks/ or the repo root.
def find_project_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "config.py").is_file() and (candidate / "app.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root (folder that contains app.py and src/)."
    )

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_DIR, EDA_PLOTS_DIR, EXPECTED_CSV_FILENAME, EXPECTED_CSV_PATH, MODEL_PLOTS_DIR, MODELS_DIR, TARGET_PLOTS_DIR
from src.data_loader import (
    identify_column_types,
    inspect_dataset,
    load_raw_dataset,
    missing_dataset_message,
    validate_dataset_file,
)
from src.eda import (
    COMPOSITE_ENGAGEMENT_COLUMN,
    DEFAULT_BOXPLOT_COLUMNS,
    LIKERT_ENGAGEMENT_COLUMNS,
    categorical_columns,
    constant_columns,
    correlation_matrix,
    generate_eda_findings,
    identifier_columns,
    iqr_outlier_table,
    numeric_columns,
    plot_boxplots,
    plot_correlation_heatmap,
    plot_count_distribution,
    plot_engagement_by_category,
    plot_engagement_subheatmap,
    plot_income_vs_engagement,
    plot_likert_distribution,
    plot_mean_engagement_by_group,
    plot_numeric_distribution,
    save_eda_findings,
    set_plot_style,
    strongest_correlations,
    verify_satisfaction_index,
)
from src.target import (
    ENGAGEMENT_INPUT_COLUMNS,
    LEVEL_COLUMN,
    SCORE_COLUMN,
    TARGET_DISCLAIMER,
    add_engagement_target,
    generate_target_notes,
    plot_engagement_level_distribution,
    plot_engagement_level_percent,
    plot_engagement_score_distribution,
    required_columns_report,
    save_target_notes,
    target_distribution_table,
)
from src.modeling import run_classification_experiment
from src.analysis import run_result_analysis
from src.interpretability import run_interpretability
from src.preprocess import feature_exclusion_columns, select_model_features

%matplotlib inline

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
set_plot_style()

print("Libraries imported.")
print("Expected CSV:", EXPECTED_CSV_PATH)
print("EDA plots:", EDA_PLOTS_DIR)
print("Target plots:", TARGET_PLOTS_DIR)
print("Model plots:", MODEL_PLOTS_DIR)
print("Saved models:", MODELS_DIR)

## 2. Validate and load the CSV

If the file is missing, the next cells print where it belongs and skip plots.
Loading uses `pandas.read_csv` only. We keep a working **copy** in memory.

In [ ]:
print("Files in data/:")
for item in sorted(DATA_DIR.iterdir()):
    print(" -", item.name)

status = validate_dataset_file()
print()
print(status["message"])

df = None
try:
    df = load_raw_dataset()
    print("Loaded a working copy. Original CSV was not modified.")
except FileNotFoundError as error:
    print(error)

## 3. Dataset dimensions

This cell reports how large the table is. Row and column counts are computed from the loaded DataFrame.

In [ ]:
if df is None:
    print(missing_dataset_message())
else:
    report = inspect_dataset(df)
    print("Number of rows:", report["n_rows"])
    print("Number of columns:", report["n_columns"])
    print()
    print("Column names:")
    for name in report["column_names"]:
        print(" -", name)
    print()
    print("Data types:")
    display(report["data_types"].rename("dtype").to_frame())
    print()
    types = identify_column_types(df)
    print("Numeric columns (pandas dtype):", types["numerical_columns"])
    print()
    print("Categorical columns (object/category/bool):", types["categorical_columns"])
    print()
    print("Constant columns:", constant_columns(df))
    print("Identifier-like columns:", identifier_columns(df))
    display(df.head())

## 4. Missing values and duplicate records

Missing-value counts and duplicate-row counts are pandas calculations on the working copy.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    missing = df.isna().sum()
    print("Missing values per column:")
    display(missing.to_frame("missing_count"))
    print("Total missing cells:", int(missing.sum()))
    print("Duplicate rows:", int(df.duplicated().sum()))
    print()
    print("Numeric summary:")
    display(df.describe(include="number").T)
    cat_cols = categorical_columns(df)
    if cat_cols:
        print("Categorical summary:")
        display(df[cat_cols].describe().T)

## 5. Numerical feature distributions

Histograms show how a numeric variable is spread (center, skew, long tails).
They do not by themselves explain *why* a value is common or rare.

### Age distribution

**What this investigates:** the age mix of employees in the file (range, typical ages, whether the sample is concentrated in one band).

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_numeric_distribution(
        df,
        "Age",
        title="Distribution of employee age",
        xlabel="Age (years)",
        filename="age_distribution.png",
    )
    print(df["Age"].describe())

### Monthly income distribution

**What this investigates:** how monthly pay is spread, including whether a long right tail of high earners is present.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_numeric_distribution(
        df,
        "MonthlyIncome",
        title="Distribution of monthly income",
        xlabel="MonthlyIncome",
        filename="monthly_income_distribution.png",
    )
    print(df["MonthlyIncome"].describe())
    print("Skewness:", round(df["MonthlyIncome"].skew(), 3))

### Years at company distribution

**What this investigates:** tenure at the current company — how many employees are new versus long-tenured.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_numeric_distribution(
        df,
        "YearsAtCompany",
        title="Distribution of years at company",
        xlabel="YearsAtCompany",
        filename="years_at_company_distribution.png",
    )
    print(df["YearsAtCompany"].describe())
    print("Skewness:", round(df["YearsAtCompany"].skew(), 3))

## 6. Engagement-related score distributions

These fields are treated as **engagement-related** in this case study because they measure satisfaction, involvement, environment, relationships, and work-life balance.
Scores are plotted as counts. They are not yet mapped to Low / Medium / High classes.

### Job satisfaction distribution

**What this investigates:** how employees are spread across the JobSatisfaction survey scale in this file.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_likert_distribution(
        df,
        "JobSatisfaction",
        title="Job satisfaction score counts",
        filename="job_satisfaction_distribution.png",
    )
    print(df["JobSatisfaction"].value_counts().sort_index())

### Job involvement distribution

**What this investigates:** how often each JobInvolvement score appears. Involvement is related to engagement but is a separate column from satisfaction.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_likert_distribution(
        df,
        "JobInvolvement",
        title="Job involvement score counts",
        filename="job_involvement_distribution.png",
    )
    print(df["JobInvolvement"].value_counts().sort_index())

### Environment satisfaction distribution

**What this investigates:** ratings of the work environment on the survey scale used in this dataset.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_likert_distribution(
        df,
        "EnvironmentSatisfaction",
        title="Environment satisfaction score counts",
        filename="environment_satisfaction_distribution.png",
    )
    print(df["EnvironmentSatisfaction"].value_counts().sort_index())

### Relationship satisfaction distribution

**What this investigates:** how relationship-satisfaction scores are distributed across employees.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_likert_distribution(
        df,
        "RelationshipSatisfaction",
        title="Relationship satisfaction score counts",
        filename="relationship_satisfaction_distribution.png",
    )
    print(df["RelationshipSatisfaction"].value_counts().sort_index())

### Work-life balance distribution

**What this investigates:** how WorkLifeBalance scores are spread, including whether mid-scale scores dominate.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_likert_distribution(
        df,
        "WorkLifeBalance",
        title="Work-life balance score counts",
        filename="work_life_balance_distribution.png",
    )
    print(df["WorkLifeBalance"].value_counts().sort_index())

## 7. Categorical feature distributions

Bar charts show headcount by category. Uneven group sizes matter later: small groups (for example a small department) make mean scores less stable.

### Overtime distribution

**What this investigates:** how many employees are recorded as working overtime versus not.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_count_distribution(
        df,
        "OverTime",
        title="Overtime status counts",
        xlabel="OverTime",
        filename="overtime_distribution.png",
    )
    print(df["OverTime"].value_counts())

### Department distribution

**What this investigates:** how the workforce is split across departments in this sample.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_count_distribution(
        df,
        "Department",
        title="Department headcount",
        xlabel="Department",
        filename="department_distribution.png",
    )
    print(df["Department"].value_counts())

### Job role distribution

**What this investigates:** which job roles are common or rare. Rare roles should not be over-interpreted in later group comparisons.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_count_distribution(
        df,
        "JobRole",
        title="Job role headcount",
        xlabel="JobRole",
        filename="job_role_distribution.png",
        horizontal=True,
    )
    print(df["JobRole"].value_counts())

## 8. Outliers

Box plots mark the median, quartiles, and points outside the typical 1.5×IQR fences.
**Flagged points are not automatically errors.** Income and tenure often have a real long tail.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    outlier_table = iqr_outlier_table(df)
    print("IQR outlier screening (numeric columns):")
    display(outlier_table)
    plot_boxplots(
        df,
        DEFAULT_BOXPLOT_COLUMNS,
        title="Box plots for selected numerical variables",
        filename="numerical_boxplots.png",
    )

## 9. Correlations between numerical variables

The heatmap uses **Pearson r**. A large r means two columns move together linearly in this sample.
It does **not** mean one column causes the other. Some pairs (job level and income, tenure fields) are structurally related.

### Correlation heatmap

**What this investigates:** pairwise linear association among numeric columns after dropping constants and identifier fields.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_correlation_heatmap(df, filename="correlation_heatmap.png")
    corr = correlation_matrix(df)
    print("Strongest absolute Pearson pairs:")
    display(strongest_correlations(corr, top_n=15))

### Engagement-related correlation heatmap

**What this investigates:** how the Likert engagement fields relate to each other and to `OverallSatisfactionIndex`.
If the index is a mean of some of those fields, large r values with those inputs are expected by construction.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    print(verify_satisfaction_index(df))
    plot_engagement_subheatmap(df, filename="engagement_correlation_heatmap.png")
    cols = [c for c in LIKERT_ENGAGEMENT_COLUMNS + [COMPOSITE_ENGAGEMENT_COLUMN] if c in df.columns]
    display(df[cols].corr())

## 10. Relationships between HR variables and engagement-related variables

The plots below compare group patterns and pairwise spread.
Read them as **associations in this dataset**, not as proof that a policy change would move engagement.

### Overall satisfaction by overtime

**What this investigates:** whether the composite satisfaction index is distributed differently for employees with overtime versus without.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_engagement_by_category(
        df,
        category="OverTime",
        engagement_col=COMPOSITE_ENGAGEMENT_COLUMN,
        title="Overall satisfaction index by overtime status",
        filename="overall_satisfaction_by_overtime.png",
    )
    display(df.groupby("OverTime")[COMPOSITE_ENGAGEMENT_COLUMN].describe())

### Overall satisfaction by department

**What this investigates:** whether department groups differ in the spread and center of the composite satisfaction index.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_engagement_by_category(
        df,
        category="Department",
        engagement_col=COMPOSITE_ENGAGEMENT_COLUMN,
        title="Overall satisfaction index by department",
        filename="overall_satisfaction_by_department.png",
    )
    display(df.groupby("Department")[COMPOSITE_ENGAGEMENT_COLUMN].describe())

### Overall satisfaction by job role

**What this investigates:** role-level differences in the composite index. Small roles have fewer people, so their boxes are less stable.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_engagement_by_category(
        df,
        category="JobRole",
        engagement_col=COMPOSITE_ENGAGEMENT_COLUMN,
        title="Overall satisfaction index by job role",
        filename="overall_satisfaction_by_job_role.png",
    )
    display(
        df.groupby("JobRole")[COMPOSITE_ENGAGEMENT_COLUMN]
        .agg(["count", "mean", "median"])
        .sort_values("mean")
    )

### Mean engagement-related scores by attrition

**What this investigates:** whether people who left (`Attrition` = Yes) have different average survey scores than people who stayed.
Attrition is an HR outcome in this file; a gap in means is still not a causal proof.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_mean_engagement_by_group(
        df,
        group_col="Attrition",
        value_cols=LIKERT_ENGAGEMENT_COLUMNS + [COMPOSITE_ENGAGEMENT_COLUMN],
        title="Mean engagement-related scores by attrition",
        filename="mean_engagement_by_attrition.png",
    )
    display(df.groupby("Attrition")[LIKERT_ENGAGEMENT_COLUMNS + [COMPOSITE_ENGAGEMENT_COLUMN]].mean())

### Work-life balance by overtime

**What this investigates:** whether WorkLifeBalance scores differ by overtime status.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_engagement_by_category(
        df,
        category="OverTime",
        engagement_col="WorkLifeBalance",
        title="Work-life balance by overtime status",
        filename="work_life_balance_by_overtime.png",
    )
    display(pd.crosstab(df["OverTime"], df["WorkLifeBalance"], normalize="index").round(3))

### Monthly income vs overall satisfaction index

**What this investigates:** whether higher pay lines up with a higher composite satisfaction index in this sample.
A flat cloud means little linear association, not that pay is unimportant in general.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_income_vs_engagement(df, filename="income_vs_overall_satisfaction.png")
    r = df["MonthlyIncome"].corr(df[COMPOSITE_ENGAGEMENT_COLUMN])
    print("Pearson r (MonthlyIncome, OverallSatisfactionIndex):", round(float(r), 3))

### Mean Likert scores by department

**What this investigates:** department differences across the five engagement-related Likert items at once.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    plot_mean_engagement_by_group(
        df,
        group_col="Department",
        value_cols=LIKERT_ENGAGEMENT_COLUMNS,
        title="Mean Likert engagement-related scores by department",
        filename="mean_likert_by_department.png",
    )
    display(df.groupby("Department")[LIKERT_ENGAGEMENT_COLUMNS].mean())

## 11. EDA Findings

The next cell **writes this section from the DataFrame**. It does not paste invented metrics.
A copy is also saved to `reports/eda_findings.md`.

In [ ]:
if df is None:
    print("EDA Findings skipped because the CSV was not loaded.")
    print(missing_dataset_message())
else:
    findings = generate_eda_findings(df)
    saved = save_eda_findings(findings)
    print("Wrote:", saved)
    # Markdown display so the generated section appears in the notebook
    from IPython.display import Markdown, display as ipy_display
    ipy_display(Markdown(findings))

## 12. Why we construct a target (the dataset has no engagement label)

The IBM HR Analytics Employee Attrition & Performance table is built around
**attrition**, pay, tenure, and several satisfaction / involvement survey items.
It does **not** ship with a field called Employee Engagement, and it does not
provide official Low / Medium / High engagement classes.

This case study still needs a 3-class target. The approach used here is a
**transparent student-defined operationalization**:

1. Treat five existing 1–4 items as related to how engaged someone reports feeling
   at work (involvement, job satisfaction, environment, relationships, work-life balance).
2. Average them with **equal weights** so every item counts the same and anyone can
   recompute the score from the raw columns.
3. Cut that mean into Low / Medium / High with **project-defined** numeric thresholds.

This is a modeling convenience for coursework. It is **not** an official HR metric,
**not** a validated psychological instrument, and **not** a claim that IBM or any
employer uses these cut-points.

`OverallSatisfactionIndex` (if present) is a different column already in the file.
Our `EngagementScore` is rebuilt on purpose so that **JobInvolvement is included**,
and so the rule lives in this project's code instead of depending on a pre-made index.

## 13. Verify the five required columns

Construction stops if any required name is missing. No substitute columns are invented.

In [ ]:
if df is None:
    print("Skipped: CSV not loaded.")
else:
    column_check = required_columns_report(df)
    print("Required columns:")
    for name in column_check["required"]:
        print(" -", name)
    print()
    print("Present:", column_check["present"])
    print("Missing:", column_check["missing"])
    print("All required columns found:", column_check["all_present"])
    if column_check["all_present"]:
        print()
        print("Value ranges (should typically be 1–4):")
        display(df[ENGAGEMENT_INPUT_COLUMNS].describe().T[["min", "max", "mean"]])
        print("Missing values in these columns:")
        print(df[ENGAGEMENT_INPUT_COLUMNS].isna().sum())

## 14. Build EngagementScore and EngagementLevel

Formula used for every employee:

`EngagementScore = mean(JobInvolvement, JobSatisfaction, EnvironmentSatisfaction, RelationshipSatisfaction, WorkLifeBalance)`

Project-defined bins (not an official HR standard):

| EngagementScore | EngagementLevel |
| --- | --- |
| 1.00–2.00 | Low |
| 2.01–3.00 | Medium |
| 3.01–4.00 | High |

The working table is a **copy**. The five original variables stay in place.
The CSV in `data/` is not saved over.

In [ ]:
labeled_df = None

if df is None:
    print("Skipped: CSV not loaded.")
else:
    print(TARGET_DISCLAIMER)
    print()
    labeled_df = add_engagement_target(df)
    print("Working copy shape before target columns:", df.shape)
    print("Working copy shape after target columns:", labeled_df.shape)
    print("Original five columns still present:", all(c in labeled_df.columns for c in ENGAGEMENT_INPUT_COLUMNS))
    print("New columns:", SCORE_COLUMN, "and", LEVEL_COLUMN)
    print()
    print("Audit sample (inputs + new columns only):")
    display(
        labeled_df[ENGAGEMENT_INPUT_COLUMNS + [SCORE_COLUMN, LEVEL_COLUMN]].head(10)
    )
    print()
    print("The original `df` object was not given the new columns (construction used a copy):")
    print(SCORE_COLUMN in df.columns, LEVEL_COLUMN in df.columns)

### Target class counts and percentages

**What this investigates:** how many employees fall into Low, Medium, and High
under the student-defined rule, and whether one class dominates.

In [ ]:
if labeled_df is None:
    print("Skipped: target was not constructed.")
else:
    summary = target_distribution_table(labeled_df)
    display(summary["table"])
    print("Total employees labeled:", summary["total"])
    print()
    print("Majority class:", summary["majority_level"],
          "n =", summary["majority_count"],
          f"({summary['majority_percent']:.2f}%)")
    print("Smallest class:", summary["minority_level"],
          "n =", summary["minority_count"])
    print("Majority-to-minority ratio:",
          summary["imbalance_ratio_majority_to_minority"])
    print("Classes are imbalanced (majority share >= 50% or ratio >= 2):",
          summary["classes_are_imbalanced"])

### EngagementScore distribution

**What this investigates:** how the continuous mean score is spread, and where
the project cut-points (2.00 and 3.00) sit relative to that spread.

In [ ]:
if labeled_df is None:
    print("Skipped: target was not constructed.")
else:
    plot_engagement_score_distribution(labeled_df)
    print(labeled_df[SCORE_COLUMN].describe())

### EngagementLevel distribution

**What this investigates:** headcount in each Low / Medium / High bin.
A tall Medium bar with short Low/High bars is a class-imbalance warning for later modeling.

In [ ]:
if labeled_df is None:
    print("Skipped: target was not constructed.")
else:
    plot_engagement_level_distribution(labeled_df)
    plot_engagement_level_percent(labeled_df)

## 15. Target construction notes

The next cell writes this section from the labeled table (counts are not typed by hand).
A copy is saved to `reports/target_construction.md`.

In [ ]:
if labeled_df is None:
    print("Target notes skipped.")
else:
    notes = generate_target_notes(labeled_df)
    saved = save_target_notes(notes)
    print("Wrote:", saved)
    from IPython.display import Markdown, display as ipy_display
    ipy_display(Markdown(notes))

## 16. Classification experiments

Target: **`EngagementLevel`** (Low / Medium / High), the student-defined label from Section 14.

Four classifiers are trained on **one shared stratified train/test split**
(`test_size=0.20`, `random_state=42`):

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Support Vector Machine

### Preprocessing

`src/preprocess.py` builds a sklearn `ColumnTransformer`:

- numeric columns → `StandardScaler`
- categorical columns → `OneHotEncoder(handle_unknown="ignore")`

The preprocessor is **fit on training rows only**, then applied to train and test.

### Leakage control

`EngagementLevel` is a function of five survey items. Those items, `EngagementScore`,
and `OverallSatisfactionIndex` are **not** used as features. IDs and constant columns
are also dropped. Otherwise a model could reconstruct the label and look perfect
without learning other HR factors.

### Class imbalance

Medium is the majority class. This experiment uses:

- **stratified** splitting so Low / Medium / High all appear in train and test
- `class_weight="balanced"` so rare classes get more weight

Rows are **not** copied (no random oversampling).

### Metrics

For this **multiclass, imbalanced** problem:

- **Accuracy** can look high if the model mostly predicts Medium.
- **Macro precision, recall, and F1** average the three classes equally, so the
  small Low class counts as much as Medium.
- **Weighted** averages follow class size and stay closer to accuracy.

**Do not pick a winner from accuracy alone.** Macro F1 is the primary comparison
metric here. No ranking is stated until the test scores below are computed.

### Feature table used for modeling

**What this investigates:** which columns enter X, and which are held out so the label cannot leak.

In [ ]:
if labeled_df is None:
    print("Modeling skipped: build EngagementLevel in Section 14 first.")
    run_result = None
else:
    X_preview, dropped_preview = select_model_features(labeled_df)
    print("Columns excluded from the model:")
    for name in dropped_preview:
        print(" -", name)
    print()
    print("Predictor shape (rows, columns):", X_preview.shape)
    print("Numeric columns:", X_preview.select_dtypes(include="number").columns.tolist())
    print("Categorical columns:", X_preview.select_dtypes(include=["object", "category", "bool"]).columns.tolist())

### Train the four models on the same split

This cell fits the preprocessor and the four classifiers, scores them on the
**test set only**, writes confusion matrices, and saves Joblib pipelines under `models/`.

In [ ]:
if labeled_df is None:
    print("Modeling skipped.")
    run_result = None
else:
    run_result = run_classification_experiment(labeled_df, save_artifacts=True)
    info = run_result["split_info"]
    print("Train rows:", info["n_train"], "| Test rows:", info["n_test"])
    print("random_state:", info["random_state"], "| test_size:", info["test_size"])
    print("Train class counts:", info["train_class_counts"])
    print("Test class counts:", info["test_class_counts"])
    print()
    print("Saved preprocessor:", MODELS_DIR / "preprocessor.joblib")
    print("Saved full pipelines: logistic_regression, decision_tree, random_forest, svm")

### Comparison table (actual test-set scores)

`precision`, `recall`, and `f1_score` in this table are **macro-averaged**
(same numbers as `precision_macro`, `recall_macro`, `f1_macro`).
Weighted averages are included so accuracy is not the only size-sensitive metric.

In [ ]:
if run_result is None:
    print("No comparison table: models were not trained.")
else:
    comparison = run_result["comparison"]
    display(comparison)
    print("CSV copy:", (PROJECT_ROOT / "reports" / "model_comparison.csv").resolve())

### Confusion matrices and classification reports

Each confusion matrix is **test-set only**. Rows are true labels; columns are predictions.

The classification report shows per-class precision, recall, F1, and support.

In [ ]:
if run_result is None:
    print("No reports: models were not trained.")
else:
    from IPython.display import Image, display as ipy_display
    for model_name, report_text in run_result["reports"].items():
        print("=" * 72)
        print(model_name)
        print("=" * 72)
        print(report_text)
        cm_path = run_result["confusion_paths"].get(model_name)
        if cm_path:
            ipy_display(Image(filename=cm_path))
    print("Accuracy vs macro F1 chart:")
    ipy_display(Image(filename=run_result["comparison_plot"]))

## 17. How to read these results

The next cell writes the discussion **from the comparison DataFrame**.
It ranks models by **macro F1 first**, not by accuracy.

In [ ]:
if run_result is None:
    print("Modeling notes skipped.")
else:
    from IPython.display import Markdown, display as ipy_display
    ipy_display(Markdown(run_result["notes"]))
    print("Also saved to reports/modeling_notes.md")

## 18. What was saved

Joblib files in `models/` (full sklearn pipelines: preprocess + classifier):

- `preprocessor.joblib`
- `logistic_regression.joblib`
- `decision_tree.joblib`
- `random_forest.joblib`
- `svm.joblib`

Use the same pipeline object later for Streamlit predictions so test encoding matches training.

`EngagementLevel` remains a **case-study operationalization**, not an official engagement standard.

## 19. Analyze the saved experiment (do not change the metrics)

This section **reads** `reports/model_comparison.csv` and scores the Joblib
files already written in Section 16. It does not retrain and it does not
overwrite those accuracy / precision / recall / F1 numbers.

Selection rule (transparent):

1. Highest **macro F1** on the test set
2. Then weighted F1, then accuracy
3. Do not select a model whose accuracy only matches the majority baseline
   while Low and High recall stay at zero

In [ ]:
analysis = run_result_analysis(save_outputs=True)
print("Loaded comparison CSV (unchanged metrics):")
display(analysis["comparison"])
print()
print("Accuracy chart:", analysis["accuracy_plot"])
print("Macro F1 chart:", analysis["macro_f1_plot"])
print("Final model copy:", analysis["final_model_path"])

### Accuracy comparison chart

**What this investigates:** test accuracy by model. A high bar can still mean
"always predict Medium." The dashed majority level is 0.738 on this split.

In [ ]:
from IPython.display import Image, Markdown, display as ipy_display
if analysis is None:
    print("Analysis was not run.")
else:
    ipy_display(Image(filename=str(analysis["accuracy_plot"])))

### Macro F1 comparison chart

**What this investigates:** whether each model recovers all three classes equally.
This is the primary selection metric.

In [ ]:
if analysis is not None:
    ipy_display(Image(filename=str(analysis["macro_f1_plot"])))

### Confusion matrix for each model

Rows = true `EngagementLevel`, columns = predicted. Same 294 test employees.

In [ ]:
if analysis is not None:
    for name, path in analysis["cm_paths"].items():
        print(name)
        display(analysis["cms"][name])
        ipy_display(Image(filename=str(path)))
        print()

### Classification report for the selected final model

The selected model is the **Decision Tree**. The report below is the file saved
during training (`reports/classification_reports/decision_tree.txt`).

In [ ]:
if analysis is not None:
    print(analysis["final_report_text"])

### Error analysis and model choice

The next cell prints a write-up built from the confusion counts and the saved
metrics. It states limitations and that association is not causation.

In [ ]:
if analysis is not None:
    ipy_display(Markdown(analysis["notes"]))
    print("Wrote reports/model_analysis.md")

## 20. Model-derived feature importance (final Decision Tree)

The selected model is a Decision Tree, so we use sklearn **Gini importance**
(`feature_importances_`: mean decrease in impurity across splits).

This is **model-derived importance**. A high bar means that feature was an
**important predictive feature** for **this tree's predictions**. It does **not**
mean the variable causes employee engagement.

The chart uses the already saved `models/final_model.joblib` pipeline. The tree is
not retrained here.

In [ ]:
importance_result = run_interpretability(top_n=15, save_outputs=True)
print("Pipeline:", importance_result["pipeline_path"])
print()
print("Top 15 important predictive features:")
display(importance_result["top"])

### Horizontal bar chart (top 15)

**What this investigates:** which transformed features were most associated with
the Decision Tree's splits, ranked by Gini importance.

In [ ]:
from IPython.display import Image, Markdown, display as ipy_display
if importance_result is not None:
    ipy_display(Image(filename=str(importance_result["plot_path"])))

### Short interpretation

The next cell writes notes from the **calculated** importance table only.

In [ ]:
if importance_result is not None:
    ipy_display(Markdown(importance_result["notes"]))
    print("Wrote", importance_result["md_path"])
    print("Full table CSV:", importance_result["csv_path"])